# ISS decoding with Graph-ISS

This notebook runs Graph-ISS as a joint candidate detector and graph-based barcode decoder. It uses the standard ISS registration and channel-normalization path, then lets Graph-ISS apply its own white-top-hat candidate filter. It bypasses Starfish/Spotiflow spot detection and Starfish/PoSTcode decoding.

## Environment

From the repository root, install all decoder integrations with:

```bash
python -m pip install -U "./ISS_decoding[postcode,spotiflow,istdeco,bardensr,graphiss]"
```

Graph-ISS itself needs neither PyTorch nor TensorFlow. The modern fork uses NumPy to run the original trained 5 x 5 signal classifier.

In [ ]:
from importlib.metadata import version
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from ISS_decoding import SpaceTx_format as STX
from ISS_decoding import decoding as DEC

print("Graph-ISS:", version("graph-iss"))
print("ISS_decoding:", version("ISS_decoding"))

## Paths and experiment layout

`EXPERIMENT_ROOT` must contain `R1`, `R2`, ... directories. Leave `CREATE_SPACETX=False` when `decoding/1_SpaceTX_format/experiment.json` already exists beneath each region.

In [ ]:
EXPERIMENT_ROOT = Path("/mnt/DATA/path/to/experiment")
CODEBOOK_CSV = Path("/mnt/DATA/path/to/codebook.csv")
OUTPUT_ROOT = None  # or Path("/mnt/DATA/path/to/decoding_outputs")
REGIONS_TO_PROCESS = [1]

CREATE_SPACETX = False
RUN_DECODING = False

PIXEL_TO_UM = 1.0
CHANNELS = ["DAPI", "Cy3", "Cy5", "AF750", "AF488"]
DECODING_CHANNELS = ["AF750", "Cy5", "Cy3", "AF488"]
NUCLEI_CHANNEL = "DAPI"
USE_CARE_IMAGES = False

## Optional: create SpaceTx

Run this only when SpaceTx files have not already been generated. Graph-ISS does not need a second image or codebook format: the adapter reads both directly from SpaceTx.

In [ ]:
if CREATE_SPACETX:
    STX.make_spacetx_format(
        input_dir=EXPERIMENT_ROOT,
        codebook_csv=CODEBOOK_CSV,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=OUTPUT_ROOT,
        pixel_to_um=PIXEL_TO_UM,
        channels=CHANNELS,
        DO_decorators=DECODING_CHANNELS,
        nuclei_channel=NUCLEI_CHANNEL,
        CARE=USE_CARE_IMAGES,
    )
else:
    print("SpaceTx creation is disabled; existing SpaceTx files will be used.")

## Graph-ISS settings

The defaults follow the upstream example: `h=0.05`, graph radius 3 px, transition radius 4 px, and spatial decay 0.33. Start without final quality cutoffs on one representative region, inspect the distributions below, and then set experiment-specific thresholds. `prior` returns codebook-compatible paths; `blind` additionally returns unexpected sequences for diagnostics.

In [ ]:
GRAPHISS_KWARGS = {
    "h": 0.05,
    "radius": 3,
    "candidate_probability_threshold": None,
    "graph_radius": 3.0,
    "transition_radius": 4.0,
    "spatial_decay": 0.33,
    "search_mode": "prior",  # change to "blind" for unexpected sequences
    "quality_distance_scale": 3.0,
    "quality_threshold": None,
    "min_signal_probability": None,
    "max_distance": None,
    "normalize_frames": True,
    "z_projection": "max",
}

PIPELINE_KWARGS = {
    "register": False,
    "register_dapi": False,
    "masking_radius": 15,  # retained for API consistency; Graph-ISS uses radius above
    "normalization_method": "MH",
}

In [ ]:
if RUN_DECODING:
    DEC.process_experiment(
        input_dir=EXPERIMENT_ROOT,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=OUTPUT_ROOT,
        decode_mode="GRAPHISS",
        graphiss_kwargs=GRAPHISS_KWARGS,
        **PIPELINE_KWARGS,
    )
else:
    print("Decoding is disabled. Set RUN_DECODING = True when ready.")

## Inspect and filter the result

Parquet is canonical and CSV is written alongside it. Graph-ISS retains the signal confidence and spatial consistency separately. The combined `graphiss_quality` is not a calibrated probability, so select cutoffs from a pilot region rather than treating it as a universal threshold.

In [ ]:
RESULT_REGION = "R1"
base = OUTPUT_ROOT if OUTPUT_ROOT is not None else EXPERIMENT_ROOT
result_dir = base / RESULT_REGION / "decoding" / "2_decoded_graphiss"
result_path = result_dir / f"{RESULT_REGION}_decoded_graphiss.parquet"

if result_path.exists():
    decoded = pd.read_parquet(result_path)
    display(decoded.head())
    print(f"{len(decoded):,} decoded paths")
    print(decoded["assignment_class"].value_counts(dropna=False))
else:
    print("No result yet:", result_path)

In [ ]:
if result_path.exists() and len(decoded):
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    axes[0].hist(decoded["graphiss_signal_probability_min"], bins=80)
    axes[0].set(title="Weakest round", xlabel="minimum signal probability")
    axes[1].hist(decoded["graphiss_max_distance"], bins=80)
    axes[1].set(title="Path spread", xlabel="maximum distance (px)")
    axes[2].hist(decoded["graphiss_quality"], bins=80)
    axes[2].set(title="Combined quality", xlabel="Graph-ISS quality")
    axes[3].scatter(decoded["x"], decoded["y"], s=1, alpha=0.5)
    axes[3].invert_yaxis()
    axes[3].set(title="Decoded paths", xlabel="x (pixels)", ylabel="y (pixels)")
    plt.tight_layout()

### Example post-decoding filter

Tune all three values from the distributions and spatial appearance in your pilot data. This example keeps known codebook targets, requires every round to look signal-like, and limits round-to-round registration spread.

In [ ]:
if result_path.exists():
    high_confidence = decoded[
        decoded["candidate_target"].notna()
        & ~decoded["assignment_class"].eq("unexpected_sequence")
        & (decoded["graphiss_signal_probability_min"] >= 0.5)
        & (decoded["graphiss_max_distance"] <= 3.0)
        & (decoded["graphiss_quality"] >= 1.0)
    ].copy()
    print(f"Kept {len(high_confidence):,} / {len(decoded):,} paths")
    display(high_confidence.head())

## Reproducibility

Each productive run writes XML and JSON manifests containing the effective Graph-ISS settings, installed version, pinned fork commit, coordinate units, and output paths. Per-FOV Parquet files in `tiles/` are restart checkpoints.

In [ ]:
manifests = sorted(result_dir.glob("decoding_run_*.json")) if result_dir.exists() else []
if manifests:
    manifest = json.loads(manifests[-1].read_text())
    print(json.dumps(manifest, indent=2))
else:
    print("No Graph-ISS run manifest found.")